# Unit 11 - AI System Evaluation (Exercise) · **V2 material**

**Atoms:** `U11-A1` to `U11-A5` · **Runtime:** ~25 seconds · **No API keys**

## Without code

Variance ratio > 3; kappa < 0.8; biased ATE > debiased ATE; treatment fails cost guardrail; late ATE < early ATE.

## 1. The question

Complete the five simulation checks that mirror the demo notebook.

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

## 4. TODO - non-determinism

Simulate 300 control (`N(3,0.15)`) and 300 treatment (`N(3.4,0.9)`) outputs. `var_ratio` = treatment variance / control variance should exceed 3.

In [ ]:
var_ratio = None  # TODO
assert var_ratio is not None
print('Variance ratio:', round(var_ratio, 2))
assert var_ratio > 3

## 5. TODO - judge agreement

Given `scores_a` and `scores_b` below (same length), compute Cohen's `kappa` on low/mid/high buckets. Should be below 0.8.

In [ ]:
scores_a = np.array([2.5, 3.1, 3.8, 4.2, 3.0, 3.5, 2.9, 4.0])
scores_b = np.array([2.7, 2.9, 3.5, 3.9, 3.2, 3.1, 3.0, 3.7])
kappa = None  # TODO
assert kappa is not None
print('Kappa:', round(kappa, 3))
assert kappa < 0.8

## 6. TODO - biased judge

Treatment arm has longer outputs. `ate_biased` uses `score = quality + 0.03*length`; `ate_true` uses `quality` only. Assert `ate_biased > ate_true`.

In [ ]:
ate_biased = None  # TODO
ate_true = None  # TODO
assert ate_biased is not None and ate_true is not None
assert ate_biased > ate_true
print('Biased ATE:', round(ate_biased, 3), 'True ATE:', round(ate_true, 3))

## 7. TODO - guardrails

`passes_guardrail` is True only if quality gain >= 0.5 **and** cost ratio <= 1.5. Control (3.0, 1.0), treatment (4.0, 2.5) should fail.

In [ ]:
passes_guardrail = None  # TODO
assert passes_guardrail is not None
assert passes_guardrail == False
print('Passes guardrail:', passes_guardrail)

## 8. TODO - drift

Early treatment mean 3.5, late 2.7; control constant 3.0. `late_ate` should be less than `early_ate`.

In [ ]:
early_ate = None  # TODO
late_ate = None  # TODO
assert early_ate is not None and late_ate is not None
assert late_ate < early_ate
print('Early ATE:', round(early_ate, 3), 'Late ATE:', round(late_ate, 3))

**Takeaway:** Simulate before you ship model treatments. **Unit:** [V2 unit 11](../V2/units/unit-11-experimenting-with-ai-systems/README.md)

## Hints

Bucket scores with `pd.cut`. Kappa = (po-pe)/(1-pe). Drift: compare means by period.

## Spoiler

```python
ctrl = rng.normal(3.0, 0.15, 300)
trt = rng.normal(3.4, 0.9, 300)
var_ratio = np.var(trt, ddof=1) / np.var(ctrl, ddof=1)

def buckets(x):
    return pd.cut(x, [-np.inf, 2.8, 3.6, np.inf], labels=[0, 1, 2]).codes

ka, kb = buckets(scores_a), buckets(scores_b)
po = (ka == kb).mean()
pe = sum((ka == k).mean() * (kb == k).mean() for k in [0, 1, 2])
kappa = (po - pe) / (1 - pe) if pe < 1 else 0.0

n_ai = 200
arms = rng.integers(0, 2, n_ai)
quality = 3.0 + 0.4 * arms + rng.normal(0, 0.3, n_ai)
out_len = 50 + 40 * arms + rng.integers(-10, 10, n_ai)
score_biased = quality + 0.03 * out_len
ate_biased = score_biased[arms == 1].mean() - score_biased[arms == 0].mean()
ate_true = quality[arms == 1].mean() - quality[arms == 0].mean()

quality_gain = 4.0 - 3.0
cost_ratio = 2.5 / 1.0
passes_guardrail = bool(quality_gain >= 0.5 and cost_ratio <= 1.5)

early_ate = 3.5 - 3.0
late_ate = 2.7 - 3.0
```